# Simulação de Demanda de Capacidade Operacional

Este notebook projeta cenários futuros de volume de tickets e avalia o impacto no backlog acumulado.

Perguntas respondidas:
- Como se comporta o backlog em 30, 60 e 90 dias a partir de hoje?
- O que acontece se a demanda crescer 20% ou 50%?
- Qual o ponto de colapso operacional em cada cenário?
- Qual cenário exige ação imediata de capacidade?

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.data_processing import tratar_dados
from src.metrics import volume_diario_recebidos, volume_diario_resolvidos, capacidade_backlog_diaria

df = tratar_dados()

## 1. Linha de Base com Estatísticas Históricas

Calculamos as médias históricas de recebimento e resolução de tickets para definir os parâmetros da simulação.

In [ ]:
vol_rec = volume_diario_recebidos(df)
vol_res = volume_diario_resolvidos(df)
backlog = capacidade_backlog_diaria(df)

media_recebidos_dia = vol_rec['recebidos'].mean()
media_resolvidos_dia = vol_res['resolvidos'].mean()
desvio_recebidos = vol_rec['recebidos'].std()
saldo_medio_dia = backlog['saldo'].mean()

# Backlog atual: saldo acumulado no período histórico
backlog_acumulado_atual = backlog['saldo'].sum()

print("=" * 50)
print("   LINHA DE BASE — DADOS HISTÓRICOS")
print("=" * 50)
print(f"  Média diária recebidos   : {media_recebidos_dia:.2f} tickets/dia")
print(f"  Desvio padrão recebidos  : {desvio_recebidos:.2f} tickets/dia")
print(f"  Média diária resolvidos  : {media_resolvidos_dia:.2f} tickets/dia")
print(f"  Saldo médio diário       : {saldo_medio_dia:.2f} tickets/dia")
print(f"  Backlog acumulado atual  : {backlog_acumulado_atual:.0f} tickets")
print("=" * 50)

## 2. Definição dos Cenários de Simulação

Quatro cenários de demanda futura, mantendo a capacidade resolutiva atual constante.

| Cenário | Variação | Descrição |
|---------|----------|-----------|
| A       | -20%     | Queda de demanda (sazonalidade favorável) |
| B       | 0%       | Baseline (manutenção do status quo) |
| C       | +20%     | Crescimento moderado |
| D       | +50%     | Crescimento agressivo |

In [ ]:
HORIZONTE_DIAS = 90

cenarios = {
    'A: -20% demanda': media_recebidos_dia * 0.80,
    'B: Baseline':     media_recebidos_dia * 1.00,
    'C: +20% demanda': media_recebidos_dia * 1.20,
    'D: +50% demanda': media_recebidos_dia * 1.50,
}

print(f"Capacidade resolutiva diária (fixa): {media_resolvidos_dia:.2f} tickets/dia")
print(f"Horizonte de simulação: {HORIZONTE_DIAS} dias")
print()
for nome, demanda in cenarios.items():
    saldo_diario = demanda - media_resolvidos_dia
    print(f"{nome}: {demanda:.2f} tickets/dia | saldo diário: {saldo_diario:+.2f}")

## 3. Simulação do Backlog Acumulado

Para cada cenário, projetamos o backlog acumulado ao longo dos 90 dias, partindo do backlog atual.

In [ ]:
dias = np.arange(0, HORIZONTE_DIAS + 1)
resultados = {}

for nome, demanda_diaria in cenarios.items():
    saldo_diario = demanda_diaria - media_resolvidos_dia
    # Backlog acumulado: começa do backlog atual e acumula o saldo diário
    backlog_projetado = backlog_acumulado_atual + saldo_diario * dias
    resultados[nome] = backlog_projetado

df_sim = pd.DataFrame(resultados, index=dias)
df_sim.index.name = 'Dias'

# Resumo em D+30, D+60, D+90
resumo = df_sim.loc[[30, 60, 90]].T.copy()
resumo.columns = ['D+30', 'D+60', 'D+90']
resumo = resumo.round(0).astype(int)
print("Backlog acumulado projetado:")
print(resumo.to_string())

## 4. Visualização da Evolução do Backlog por Cenário

In [ ]:
cores = {
    'A: -20% demanda': '#2ca02c',
    'B: Baseline':     '#1f77b4',
    'C: +20% demanda': '#ff7f0e',
    'D: +50% demanda': '#d62728',
}
estilos = {
    'A: -20% demanda': '--',
    'B: Baseline':     '-',
    'C: +20% demanda': '-.',
    'D: +50% demanda': ':',
}

fig, ax = plt.subplots(figsize=(12, 6))

for nome, serie in resultados.items():
    ax.plot(dias, serie, label=nome, color=cores[nome],
            linestyle=estilos[nome], linewidth=2.2)

# Marcos temporais
for marco in [30, 60, 90]:
    ax.axvline(marco, color='gray', linestyle=':', linewidth=1, alpha=0.6)
    ax.text(marco + 0.5, ax.get_ylim()[0], f'D+{marco}', color='gray', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.4)
ax.set_title('Simulação de Backlog Acumulado — Próximos 90 Dias', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Dias a partir de hoje')
ax.set_ylabel('Backlog acumulado (tickets)')
ax.legend(title='Cenário', loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Ponto de Colapso Operacional

Defini como "colapso" quando o backlog acumulado dobra em relação ao valor atual. Calculado em quantos dias cada cenário atinge esse patamar.

In [ ]:
limite_critico = max(backlog_acumulado_atual * 2, backlog_acumulado_atual + 50)  # pelo menos +50 tickets

print(f"Backlog atual       : {backlog_acumulado_atual:.0f} tickets")
print(f"Limite crítico (2x) : {limite_critico:.0f} tickets")
print()
print("-" * 50)

for nome, serie in resultados.items():
    dias_critico = np.where(serie >= limite_critico)[0]
    if len(dias_critico) > 0:
        print(f"{nome:25s}: atinge limite em D+{dias_critico[0]}")
    else:
        backlog_90 = serie[-1]
        if backlog_90 < backlog_acumulado_atual:
            print(f"{nome:25s}: REDUZ backlog para {backlog_90:.0f} em D+90")
        else:
            print(f"{nome:25s}: não atinge limite em 90 dias (D+90 = {backlog_90:.0f})")

## 6. Tabela de Resumo dos Cenários

In [ ]:
linhas = []
for nome, demanda in cenarios.items():
    saldo = demanda - media_resolvidos_dia
    b30 = int(resultados[nome][30])
    b60 = int(resultados[nome][60])
    b90 = int(resultados[nome][90])
    dias_col = np.where(resultados[nome] >= limite_critico)[0]
    colapso = f"D+{dias_col[0]}" if len(dias_col) > 0 else "> 90 dias"
    linhas.append({
        'Cenário': nome,
        'Demanda/dia': f"{demanda:.1f}",
        'Saldo/dia': f"{saldo:+.1f}",
        'Backlog D+30': b30,
        'Backlog D+60': b60,
        'Backlog D+90': b90,
        'Colapso (2x)': colapso
    })

df_resumo = pd.DataFrame(linhas).set_index('Cenário')
print(df_resumo.to_string())

## 7. Conclusão da Simulação de Demanda

Resumo interpretativo dos cenários:

In [ ]:
print("=" * 60)
print("         CONCLUSÃO — SIMULAÇÃO DE DEMANDA")
print("=" * 60)
print()
print("  Cenário A (-20%): Queda de demanda alivia o backlog.")
print("  Cenário B (0%):   Status quo — backlog segue trajetória atual.")
print("  Cenário C (+20%): Crescimento moderado pressiona a capacidade.")
print("  Cenário D (+50%): Colapso iminente — ação urgente necessária.")
print()
print(f"  Capacidade diária atual: {media_resolvidos_dia:.1f} tickets")
print(f"  Demanda diária atual:    {media_recebidos_dia:.1f} tickets")
print(f"  Folga/déficit atual:     {media_resolvidos_dia - media_recebidos_dia:+.1f} tickets/dia")
print()
print("  → Consultar notebook 07 para dimensionamento de equipe")
print("    necessário em cada cenário.")
print("=" * 60)